# Multicoordinate Wave Project

Generate one wave project containing several coordinate systems and inspect the coordinate-specific artifacts.

Navigation: [Index](../index.ipynb) | Previous: [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb) | Next: [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)


## Generate and Run the Multicoordinate Project
The project contains coordinate-specific directories and diagnostics for Cartesian, Spherical, SinhCartesian, and SinhSpherical grids.

## Import Multicoordinate Project Execution Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create a Multicoordinate Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
PROJECT_NAME = "wave_equation_multicoordinates"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_multi_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME


## Generate the Multicoordinate Project

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [ ]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_multicoordinates"]
print("generator command: python -m nrpy.examples.wave_equation_multicoordinates")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)


## Step 4: Shorten runtime parameters

Only runtime values are changed so the notebook run finishes quickly.

In [ ]:
parfile = PROJECT_DIR / "wave_equation_multicoordinates.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace(
    "diagnostics_output_every = 0.2", "diagnostics_output_every = 0.1"
)
par_text = par_text.replace(
    "output_progress_every = 1", "output_progress_every = 1000000"
)
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))


## Inspect Coordinate-Specific Directories

The output shows the Runge-Kutta or Method of Lines objects used by generated code.


In [ ]:
coordinate_dirs = [
    path.name
    for path in PROJECT_DIR.iterdir()
    if path.is_dir() and path.name not in {"MoL", "diagnostics", "intrinsics"}
]
print("coordinate directories:")
for name in sorted(coordinate_dirs):
    print(name)
missing = {"Cartesian", "SinhCartesian", "SinhSpherical", "Spherical"}.difference(
    set(coordinate_dirs)
)
if missing:
    raise RuntimeError(f"Missing coordinate directories: {sorted(missing)}")


## Step 6: Inspect the generated inventory

The inventory identifies the generated files relevant to this lesson.

In [ ]:
print("coordinate directories and core files:")
for name in sorted(coordinate_dirs):
    print(name + "/")
for relative_path in ["Makefile", "BHaH_function_prototypes.h", parfile.name]:
    print(relative_path)


## Step 7: Inspect the multicoordinate parameter file

The parameter file records the coordinate systems available to the generated executable.

In [ ]:
print("\n--- wave_equation_multicoordinates.par ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))


## Step 8: Build the executable

The build step compiles generated C after checking that external build tools are available.

In [ ]:
require_toolchain()
build_output = run_command(["make", "-j2"], PROJECT_DIR, timeout=300)
print("build completed")
print("compiler output line count:", len(build_output.splitlines()))


## Step 9: Run the executable

The run produces diagnostic files that are inspected in the following cells.

In [ ]:
run_output = run_command([f"./{PROJECT_NAME}", "2.0"], PROJECT_DIR, timeout=120)
print("run output:")
for line in run_output.splitlines()[:12]:
    if line.strip():
        print(line.rstrip())


## Read Multicoordinate Diagnostics

The diagnostic rows provide the numerical evidence used for interpretation.


In [ ]:
diagnostics = sorted(PROJECT_DIR.glob("out0d-grid*.txt"))
if not diagnostics:
    raise FileNotFoundError("No diagnostic files were produced.")
for diagnostic in diagnostics:
    rows = [
        line
        for line in diagnostic.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]
    if len(rows) < 2:
        raise RuntimeError(f"Expected at least two rows in {diagnostic.name}.")
    print(diagnostic.name, "rows:", len(rows), "last row:", rows[-1])


The coordinate directories and runtime parameter file show one generated project carrying several coordinate systems. The diagnostics demonstrate that each grid writes its own output using the same executable.


## Continue to Project Anatomy
- [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb)
- [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)
